# Анализ данных и типа города

Ноутбук проходит по всем областям из `data`, собирает сводные метрики, считает корреляции и классифицирует структуру города по входящим commuting-потокам: моноцентричная, полицентричная, равномерная или смешанная.

Идея классификации: входящий OD-поток в зону интерпретируется как сила рабочего центра. Один доминирующий центр дает моноцентричный город, несколько разделенных сильных центров дают полицентричный город, а высокая энтропия и низкая концентрация дают равномерный тип.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML, display

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    warnings.warn('matplotlib is not installed. Tables will work, plot cells will show a short note.')

DATA_PATH = Path('data')
RANDOM_SEED = 42
PAIR_SAMPLE_PER_AREA = 1000
SELECTED_AREA_ID = '48201'

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Data directory not found: {DATA_PATH.resolve()}')

area_ids = sorted(path.name for path in DATA_PATH.iterdir() if path.is_dir())
print(f'Areas found: {len(area_ids)}')
print(f'First ids: {area_ids[:10]}')

Areas found: 2265
First ids: ['01001', '01003', '01005', '01007', '01009', '01013', '01015', '01017', '01019', '01021']


C:\Users\rkozl\AppData\Local\Temp\ipykernel_26228\1105495297.py:12: UserWarning: matplotlib is not installed. Tables will work, plot cells will show a short note.
  warnings.warn('matplotlib is not installed. Tables will work, plot cells will show a short note.')


## Функции анализа

Основные признаки типа города:
- `top1_inflow_share`: доля входящего потока крупнейшего центра.
- `top1_lift`: во сколько раз крупнейший центр сильнее среднего равномерного центра, то есть `top1_share * N`.
- `top1_to_top2`: доминирование первого центра над вторым.
- `inflow_entropy`: нормированная энтропия распределения входящих потоков. Ближе к 1 означает более равномерно.
- `inflow_norm_hhi`: нормированный индекс Херфиндаля-Хиршмана. Ближе к 1 означает более концентрированно.
- `center_count`: число сильных центров, разделенных по матрице расстояний.

In [2]:
TYPE_RU = {
    'monocentric': 'моноцентричный',
    'polycentric': 'полицентричный',
    'uniform': 'равномерный',
    'mixed': 'смешанный/другой',
}


def load_area(area_id, mmap_mode=None):
    area_path = DATA_PATH / str(area_id)
    return {
        'demos': np.load(area_path / 'demos.npy', mmap_mode=mmap_mode),
        'pois': np.load(area_path / 'pois.npy', mmap_mode=mmap_mode),
        'adj': np.load(area_path / 'adj.npy', mmap_mode=mmap_mode),
        'dis': np.load(area_path / 'dis.npy', mmap_mode=mmap_mode),
        'od': np.load(area_path / 'od.npy', mmap_mode=mmap_mode),
    }


def safe_shares(values):
    values = np.asarray(values, dtype=float)
    total = values.sum()
    if values.size == 0:
        return values
    if total <= 0:
        return np.full(values.shape, 1 / values.size, dtype=float)
    return values / total


def normalized_hhi(shares):
    n = len(shares)
    if n <= 1:
        return 1.0
    hhi = float(np.square(shares).sum())
    return float((hhi - 1 / n) / (1 - 1 / n))


def effective_center_count(shares):
    hhi = float(np.square(shares).sum())
    return 0.0 if hhi <= 0 else 1.0 / hhi


def normalized_entropy(shares):
    n = len(shares)
    positive = shares[shares > 0]
    if n <= 1 or positive.size == 0:
        return 0.0
    return float(-(positive * np.log(positive)).sum() / np.log(n))


def gini(values):
    values = np.sort(np.asarray(values, dtype=float))
    n = values.size
    total = values.sum()
    if n == 0 or total <= 0:
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * values) / (n * total)) - (n + 1) / n)


def find_flow_centers(inflow, dis, high_quantile=0.85, min_top_ratio=0.25, separation_quantile=0.20):
    inflow = np.asarray(inflow, dtype=float)
    dis = np.asarray(dis, dtype=float)
    if inflow.size == 0 or inflow.max(initial=0) <= 0:
        return []

    positive_inflow = inflow[inflow > 0]
    threshold = max(float(np.quantile(positive_inflow, high_quantile)), float(inflow.max() * min_top_ratio))
    candidates = np.where(inflow >= threshold)[0]
    if candidates.size == 0:
        return [int(inflow.argmax())]

    positive_distances = dis[dis > 0]
    separation = float(np.quantile(positive_distances, separation_quantile)) if positive_distances.size else 0.0

    centers = []
    for idx in sorted(candidates, key=lambda i: inflow[i], reverse=True):
        if not centers or np.min(dis[idx, centers]) >= separation:
            centers.append(int(idx))
    return centers


def classify_city_type(metrics):
    n = metrics['n_nodes']
    if n <= 3:
        return 'mixed'

    top1_share = metrics['top1_inflow_share']
    top1_lift = metrics['top1_lift']
    dominance = metrics['top1_to_top2']
    entropy = metrics['inflow_entropy']
    norm_hhi = metrics['inflow_norm_hhi']
    center_count = metrics['center_count']

    if entropy >= 0.90 and norm_hhi <= 0.04 and top1_lift <= 3.0:
        return 'uniform'

    if (
        (center_count <= 1 and top1_lift >= 4.0 and dominance >= 1.8)
        or (top1_share >= 0.35 and dominance >= 1.5)
        or (norm_hhi >= 0.20 and dominance >= 1.6)
    ):
        return 'monocentric'

    if center_count >= 2 and dominance < 1.8 and top1_lift >= 2.0:
        return 'polycentric'

    return 'mixed'


def compute_area_metrics(area_id):
    raw = load_area(area_id, mmap_mode='r')
    od = np.asarray(raw['od'], dtype=float)
    dis = np.asarray(raw['dis'], dtype=float)
    adj = np.asarray(raw['adj'], dtype=float)
    n = od.shape[0]
    total_flow = float(od.sum())
    inflow = od.sum(axis=0)
    outflow = od.sum(axis=1)
    shares = safe_shares(inflow)
    ordered = np.sort(shares)[::-1]
    centers = find_flow_centers(inflow, dis)
    positive_dis = dis[dis > 0]

    metrics = {
        'area_id': str(area_id),
        'n_nodes': int(n),
        'n_demo_features': int(raw['demos'].shape[1]),
        'n_poi_features': int(raw['pois'].shape[1]),
        'total_flow': total_flow,
        'od_density': float(np.count_nonzero(od) / od.size),
        'adj_density': float(np.count_nonzero(adj) / adj.size),
        'self_flow_share': float(np.trace(od) / total_flow) if total_flow > 0 else 0.0,
        'mean_distance': float(np.mean(positive_dis)) if positive_dis.size else 0.0,
        'median_distance': float(np.median(positive_dis)) if positive_dis.size else 0.0,
        'mean_inflow': float(np.mean(inflow)) if n else 0.0,
        'max_inflow': float(np.max(inflow)) if n else 0.0,
        'mean_outflow': float(np.mean(outflow)) if n else 0.0,
        'max_outflow': float(np.max(outflow)) if n else 0.0,
        'inflow_gini': gini(inflow),
        'outflow_gini': gini(outflow),
        'inflow_entropy': normalized_entropy(shares),
        'inflow_norm_hhi': normalized_hhi(shares),
        'effective_centers': effective_center_count(shares),
        'top1_inflow_share': float(ordered[0]) if n else 0.0,
        'top2_inflow_share': float(ordered[1]) if n > 1 else 0.0,
        'top3_inflow_share': float(ordered[:3].sum()) if n else 0.0,
        'center_count': int(len(centers)),
        'center_nodes': ','.join(map(str, centers)),
    }
    metrics['top1_to_top2'] = metrics['top1_inflow_share'] / metrics['top2_inflow_share'] if metrics['top2_inflow_share'] > 0 else np.inf
    metrics['top1_lift'] = metrics['top1_inflow_share'] * n
    metrics['city_type'] = classify_city_type(metrics)
    metrics['city_type_ru'] = TYPE_RU[metrics['city_type']]
    return metrics

## Сводка по всем областям

In [3]:
rows = []
failed = []

for area_id in area_ids:
    try:
        rows.append(compute_area_metrics(area_id))
    except Exception as exc:
        failed.append({'area_id': area_id, 'error': repr(exc)})

summary = pd.DataFrame(rows).sort_values('area_id').reset_index(drop=True)

print(f'Loaded areas: {len(summary)}')
print(f'Failed areas: {len(failed)}')
if failed:
    display(pd.DataFrame(failed).head(20))

display(summary.head())
display(summary['city_type_ru'].value_counts().to_frame('count'))

Loaded areas: 2265
Failed areas: 0


,area_id,n_nodes,n_demo_features,n_poi_features,total_flow,od_density,adj_density,self_flow_share,mean_distance,median_distance,mean_inflow,max_inflow,mean_outflow,max_outflow,inflow_gini,outflow_gini,inflow_entropy,inflow_norm_hhi,effective_centers,top1_inflow_share,top2_inflow_share,top3_inflow_share,center_count,center_nodes,top1_to_top2,top1_lift,city_type,city_type_ru
0,01001,12,97,34,4735.0,0.958333,0.361111,0.160296,14948.755551,11436.022949,394.583333,974.0,394.583333,1029.0,0.422052,0.273759,0.881693,0.056895,7.380797,0.205702,0.192186,0.568532,2,"4,1",1.070330,2.468427,polycentric,полицентричный
1,01003,31,97,34,44846.0,0.947971,0.147763,0.154596,32954.411731,30408.976562,1446.645161,4167.0,1446.645161,3346.0,0.404966,0.268044,0.921829,0.018412,19.969781,0.092918,0.084088,0.254538,2,"28,7",1.105012,2.880458,uniform,равномерный
2,01005,9,97,34,4389.0,0.962963,0.469136,0.236956,24440.643392,22496.187500,487.666667,1084.0,487.666667,966.0,0.366219,0.253108,0.895882,0.055702,6.225729,0.246981,0.220551,0.596036,2,"8,4",1.119835,2.222830,polycentric,полицентричный
3,01007,4,97,34,1610.0,1.000000,0.625000,0.543478,22038.771159,22780.616211,402.500000,785.0,402.500000,721.0,0.347516,0.289441,0.852729,0.134503,2.850002,0.487578,0.296273,0.923602,1,3,1.645702,1.950311,monocentric,моноцентричный
4,01009,9,97,34,4242.0,0.925926,0.395062,0.367280,24876.979736,23186.325195,471.333333,1417.0,471.333333,757.0,0.400073,0.243386,0.870948,0.080477,5.475057,0.334041,0.143564,0.616690,2,"1,5",2.326765,3.006365,mixed,смешанный/другой


,count
city_type_ru,
смешанный/другой,756
моноцентричный,679
полицентричный,663
равномерный,167


## Корреляции и общие визуализации

In [8]:
!pip install -q matplotlib 


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\rkozl\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
numeric_cols = [
    'n_nodes', 'total_flow', 'od_density', 'adj_density', 'self_flow_share',
    'mean_distance', 'median_distance', 'inflow_gini', 'outflow_gini',
    'inflow_entropy', 'inflow_norm_hhi', 'effective_centers',
    'top1_inflow_share', 'top2_inflow_share', 'top3_inflow_share',
    'top1_to_top2', 'top1_lift', 'center_count',
]

corr = summary[numeric_cols].replace([np.inf, -np.inf], np.nan).corr()
display(corr.round(3))

if plt is None:
    display(HTML('<b>Install matplotlib to render plots:</b> pip install matplotlib'))
else:
    fig, ax = plt.subplots(figsize=(12, 10))
    image = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticklabels(corr.index)
    ax.set_title('Correlation heatmap for area-level metrics')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    type_order = summary['city_type_ru'].value_counts().index.tolist()
    color_map = {
        'моноцентричный': '#d62728',
        'полицентричный': '#1f77b4',
        'равномерный': '#2ca02c',
        'смешанный/другой': '#7f7f7f',
    }
    colors = summary['city_type_ru'].map(color_map)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    summary['city_type_ru'].value_counts().loc[type_order].plot(kind='bar', ax=axes[0, 0], color=[color_map[t] for t in type_order])
    axes[0, 0].set_title('City type distribution')
    axes[0, 0].set_xlabel('')
    axes[0, 0].set_ylabel('Area count')

    axes[0, 1].hist(summary['n_nodes'], bins=50, color='#4c78a8')
    axes[0, 1].set_title('Number of zones per area')
    axes[0, 1].set_xlabel('N zones')
    axes[0, 1].set_ylabel('Area count')

    axes[1, 0].scatter(summary['top1_lift'], summary['inflow_entropy'], c=colors, s=18, alpha=0.75)
    axes[1, 0].set_xscale('log')
    axes[1, 0].set_title('Dominance vs entropy')
    axes[1, 0].set_xlabel('top1_lift: strongest center / uniform average')
    axes[1, 0].set_ylabel('Inflow entropy')

    axes[1, 1].scatter(summary['top1_inflow_share'], summary['top1_to_top2'].clip(upper=10), c=colors, s=18, alpha=0.75)
    axes[1, 1].set_title('Largest center share vs top1/top2 dominance')
    axes[1, 1].set_xlabel('Top-1 inflow share')
    axes[1, 1].set_ylabel('Top-1 / Top-2 ratio, clipped at 10')

    plt.tight_layout()
    plt.show()

,n_nodes,total_flow,od_density,adj_density,self_flow_share,mean_distance,median_distance,inflow_gini,outflow_gini,inflow_entropy,inflow_norm_hhi,effective_centers,top1_inflow_share,top2_inflow_share,top3_inflow_share,top1_to_top2,top1_lift,center_count
n_nodes,1.000,0.936,-0.711,-0.514,-0.418,-0.012,-0.014,0.377,0.095,0.095,-0.258,0.840,-0.405,-0.451,-0.525,-0.047,0.855,0.448
total_flow,0.936,1.000,-0.584,-0.437,-0.365,-0.009,-0.011,0.337,0.071,0.074,-0.220,0.733,-0.344,-0.385,-0.448,-0.038,0.814,0.357
od_density,-0.711,-0.584,1.000,0.644,0.384,-0.038,-0.037,-0.568,-0.397,0.076,0.165,-0.709,0.394,0.502,0.548,-0.019,-0.624,-0.522
adj_density,-0.514,-0.437,0.644,1.000,0.703,0.011,0.014,-0.461,-0.272,-0.289,0.537,-0.666,0.776,0.716,0.899,0.219,-0.460,-0.688
self_flow_share,-0.418,-0.365,0.384,0.703,1.000,0.106,0.112,-0.437,-0.031,-0.248,0.488,-0.536,0.700,0.592,0.781,0.223,-0.389,-0.573
mean_distance,-0.012,-0.009,-0.038,0.011,0.106,1.000,0.998,-0.048,0.019,0.021,-0.010,-0.017,0.014,0.032,0.030,-0.009,-0.016,-0.022
median_distance,-0.014,-0.011,-0.037,0.014,0.112,0.998,1.000,-0.050,0.018,0.020,-0.007,-0.019,0.018,0.036,0.035,-0.009,-0.018,-0.026
inflow_gini,0.377,0.337,-0.568,-0.461,-0.437,-0.048,-0.050,1.000,0.400,-0.627,0.318,0.317,0.011,-0.417,-0.259,0.308,0.466,0.312
outflow_gini,0.095,0.071,-0.397,-0.272,-0.031,0.019,0.018,0.400,1.000,-0.260,0.117,0.102,-0.033,-0.152,-0.122,0.090,0.124,0.157
inflow_entropy,0.095,0.074,0.076,-0.289,-0.248,0.021,0.020,-0.627,-0.260,1.000,-0.926,0.252,-0.737,-0.097,-0.497,-0.694,-0.049,0.257


## Выборочная корреляция по OD-парам

Для pair-level анализа берется до `PAIR_SAMPLE_PER_AREA` пар из каждой области. Это покрывает все области, но не раздувает память на больших матрицах.

In [5]:
def build_pair_sample(area_ids, max_pairs_per_area=PAIR_SAMPLE_PER_AREA, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    chunks = []

    for area_id in area_ids:
        raw = load_area(area_id, mmap_mode='r')
        od = np.asarray(raw['od'], dtype=float)
        dis = np.asarray(raw['dis'], dtype=float)
        adj = np.asarray(raw['adj'], dtype=float)
        n = od.shape[0]
        total_pairs = n * n
        k = min(max_pairs_per_area, total_pairs)
        flat_idx = rng.choice(total_pairs, size=k, replace=False)
        origins = flat_idx // n
        destinations = flat_idx % n
        flat_od = od.reshape(-1)
        flat_dis = dis.reshape(-1)
        flat_adj = adj.reshape(-1)
        inflow = od.sum(axis=0)
        outflow = od.sum(axis=1)

        chunks.append(pd.DataFrame({
            'area_id': area_id,
            'flow': flat_od[flat_idx],
            'log_flow': np.log1p(flat_od[flat_idx]),
            'distance': flat_dis[flat_idx],
            'adjacent': flat_adj[flat_idx],
            'same_zone': origins == destinations,
            'origin_outflow': outflow[origins],
            'dest_inflow': inflow[destinations],
        }))

    return pd.concat(chunks, ignore_index=True)


pair_sample = build_pair_sample(area_ids)
print(f'Pair sample rows: {len(pair_sample):,}')
display(pair_sample.head())

pair_corr_cols = ['flow', 'log_flow', 'distance', 'adjacent', 'same_zone', 'origin_outflow', 'dest_inflow']
pair_corr = pair_sample[pair_corr_cols].corr()
display(pair_corr.round(3))

if plt is None:
    display(HTML('<b>Install matplotlib to render pair-level plots:</b> pip install matplotlib'))
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    image = axes[0].imshow(pair_corr, vmin=-1, vmax=1, cmap='coolwarm')
    axes[0].set_xticks(range(len(pair_corr.columns)))
    axes[0].set_yticks(range(len(pair_corr.index)))
    axes[0].set_xticklabels(pair_corr.columns, rotation=90)
    axes[0].set_yticklabels(pair_corr.index)
    axes[0].set_title('Pair-level correlation')
    fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)

    scatter = pair_sample.sample(min(50000, len(pair_sample)), random_state=RANDOM_SEED)
    axes[1].scatter(scatter['distance'], scatter['log_flow'], s=3, alpha=0.08, color='#1f77b4')
    axes[1].set_title('Distance vs log(1 + flow), sampled pairs')
    axes[1].set_xlabel('Distance')
    axes[1].set_ylabel('log(1 + OD flow)')

    plt.tight_layout()
    plt.show()

Pair sample rows: 705,327


,area_id,flow,log_flow,distance,adjacent,same_zone,origin_outflow,dest_inflow
0,01001,74.0,4.317488,3907.157471,1.0,False,292.0,974.0
1,01001,0.0,0.000000,25655.294922,0.0,False,299.0,99.0
2,01001,64.0,4.174387,1246.215820,1.0,False,299.0,910.0
3,01001,1.0,0.693147,23871.851562,0.0,False,339.0,99.0
4,01001,202.0,5.313206,7653.329102,1.0,False,1029.0,910.0


,flow,log_flow,distance,adjacent,same_zone,origin_outflow,dest_inflow
flow,1.000,0.670,-0.058,0.224,0.355,0.177,0.371
log_flow,0.670,1.000,-0.096,0.333,0.282,0.198,0.412
distance,-0.058,-0.096,1.000,-0.059,-0.064,-0.004,-0.013
adjacent,0.224,0.333,-0.059,1.000,-0.084,-0.038,-0.003
same_zone,0.355,0.282,-0.064,-0.084,1.000,-0.040,-0.016
origin_outflow,0.177,0.198,-0.004,-0.038,-0.040,1.000,0.152
dest_inflow,0.371,0.412,-0.013,-0.003,-0.016,0.152,1.000


## Детальная диагностика выбранного города

По умолчанию выбран `48201`, как single-city область из README. Можно заменить `SELECTED_AREA_ID` в первой ячейке на любой id из `summary.area_id`.

In [6]:
def classical_mds(dis, n_components=2):
    dis = np.asarray(dis, dtype=float)
    n = dis.shape[0]
    d2 = dis ** 2
    centering = np.eye(n) - np.ones((n, n)) / n
    gram = -0.5 * centering @ d2 @ centering
    eigvals, eigvecs = np.linalg.eigh(gram)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order[:n_components]]
    eigvecs = eigvecs[:, order[:n_components]]
    return eigvecs * np.sqrt(np.maximum(eigvals, 0.0))


def plot_area_diagnostics(area_id, max_mds_nodes=800):
    raw = load_area(area_id)
    od = np.asarray(raw['od'], dtype=float)
    dis = np.asarray(raw['dis'], dtype=float)
    n = od.shape[0]
    inflow = od.sum(axis=0)
    shares = safe_shares(inflow)
    centers = find_flow_centers(inflow, dis)
    area_row = summary.loc[summary['area_id'] == str(area_id)]

    if area_row.empty:
        raise ValueError(f'Area {area_id} is not present in summary')

    display(area_row.T.rename(columns={area_row.index[0]: area_id}))
    center_table = pd.DataFrame({
        'node_id': centers,
        'inflow': inflow[centers] if centers else [],
        'inflow_share': shares[centers] if centers else [],
    }).sort_values('inflow', ascending=False)
    display(center_table)

    if plt is None:
        display(HTML('<b>Install matplotlib to render selected-area diagnostics.</b>'))
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(np.log1p(od), cmap='viridis', aspect='auto')
    axes[0].set_title(f'OD matrix log(1 + flow), area {area_id}')
    axes[0].set_xlabel('Destination zone')
    axes[0].set_ylabel('Origin zone')

    sorted_shares = np.sort(shares)[::-1]
    axes[1].plot(np.arange(1, n + 1), np.cumsum(sorted_shares), color='#d62728')
    axes[1].axhline(0.8, color='black', linestyle='--', linewidth=1)
    axes[1].set_title('Cumulative inflow concentration')
    axes[1].set_xlabel('Top-k destination zones')
    axes[1].set_ylabel('Cumulative inflow share')
    axes[1].set_ylim(0, 1.02)

    if n > max_mds_nodes:
        rng = np.random.default_rng(RANDOM_SEED)
        top_nodes = np.argsort(inflow)[::-1][:min(100, n)]
        remaining = np.setdiff1d(np.arange(n), top_nodes, assume_unique=False)
        sampled_rest = rng.choice(remaining, size=max_mds_nodes - len(top_nodes), replace=False) if len(remaining) > max_mds_nodes - len(top_nodes) else remaining
        plot_idx = np.sort(np.unique(np.concatenate([top_nodes, sampled_rest])))
    else:
        plot_idx = np.arange(n)

    coords = classical_mds(dis[np.ix_(plot_idx, plot_idx)])
    plot_inflow = inflow[plot_idx]
    sizes = 20 + 280 * safe_shares(plot_inflow)
    points = axes[2].scatter(coords[:, 0], coords[:, 1], s=sizes, c=np.log1p(plot_inflow), cmap='viridis', alpha=0.75)
    center_positions = [int(np.where(plot_idx == center)[0][0]) for center in centers if center in set(plot_idx)]
    if center_positions:
        axes[2].scatter(coords[center_positions, 0], coords[center_positions, 1], s=180, facecolors='none', edgecolors='red', linewidths=2, label='detected center')
        axes[2].legend()
    axes[2].set_title('Distance-based 2D layout, colored by inflow')
    axes[2].set_xlabel('MDS-1')
    axes[2].set_ylabel('MDS-2')
    fig.colorbar(points, ax=axes[2], label='log(1 + inflow)')

    plt.tight_layout()
    plt.show()


plot_area_diagnostics(SELECTED_AREA_ID)

,48201
area_id,48201
n_nodes,786
n_demo_features,97
n_poi_features,34
total_flow,1585214.0
od_density,0.46443
adj_density,0.00831
self_flow_share,0.023702
mean_distance,27754.677784
median_distance,26084.894531


,node_id,inflow,inflow_share
0,0,107261.0,0.067663


## Тип города для всех областей

Полная таблица классификации для каждого `area_id`. В ней оставлены не только метки, но и диагностические признаки, по которым классификатор принимает решение.

In [ ]:
city_type_columns = [
    'area_id', 'city_type_ru', 'city_type', 'n_nodes', 'total_flow',
    'center_count', 'center_nodes', 'top1_inflow_share', 'top2_inflow_share',
    'top3_inflow_share', 'top1_to_top2', 'top1_lift',
    'inflow_entropy', 'inflow_norm_hhi', 'effective_centers',
]

city_types_all = (
    summary[city_type_columns]
    .sort_values(['city_type_ru', 'area_id'])
    .reset_index(drop=True)
)

city_type_stats = (
    city_types_all
    .groupby('city_type_ru')
    .agg(
        cities=('area_id', 'count'),
        median_nodes=('n_nodes', 'median'),
        median_total_flow=('total_flow', 'median'),
        median_center_count=('center_count', 'median'),
        median_top1_lift=('top1_lift', 'median'),
        median_entropy=('inflow_entropy', 'median'),
        median_norm_hhi=('inflow_norm_hhi', 'median'),
    )
    .sort_values('cities', ascending=False)
)

display(city_type_stats.round(3))
display(HTML(
    '<div style="max-height: 520px; overflow: auto; border: 1px solid #ddd; padding: 8px">'
    + city_types_all.to_html(index=False)
    + '</div>'
))

print(f'City type labels assigned for {len(city_types_all)} areas.')

## Сохранение результатов

In [7]:
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)

summary_path = output_dir / 'data_city_type_summary.csv'
city_types_path = output_dir / 'data_city_types_all.csv'
corr_path = output_dir / 'data_city_metric_correlations.csv'
pair_corr_path = output_dir / 'data_pair_sample_correlations.csv'

summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
city_types_all.to_csv(city_types_path, index=False, encoding='utf-8-sig')
corr.to_csv(corr_path, encoding='utf-8-sig')
pair_corr.to_csv(pair_corr_path, encoding='utf-8-sig')

print(f'Saved summary: {summary_path}')
print(f'Saved all city types: {city_types_path}')
print(f'Saved area-level correlations: {corr_path}')
print(f'Saved pair-level correlations: {pair_corr_path}')

Saved summary: results\data_city_type_summary.csv
Saved all city types: results\data_city_types_all.csv
Saved area-level correlations: results\data_city_metric_correlations.csv
Saved pair-level correlations: results\data_pair_sample_correlations.csv
